In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
import torch
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" # Pour voir l'erreur exacte si ça plante
torch.backends.cudnn.benchmark = False   # Désactive l'auto-optimisation qui peut figer au début

In [ ]:
import os
import sys
import time
import json
import datetime
import argparse

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg") 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# C'est ici qu'on ajoute les bibliothèques de deep learning
import segmentation_models_pytorch as smp
import albumentations as A
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
import torch

In [ ]:
import os
import torch
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class TrainConfig:
    """Configuration d'entraînement fixe pour Colab (Haute Résolution)."""

    # ── Chemins ──────────────────────────────────────────────────────
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    mask_dir: str = ""
    output_dir: str = ""

    # ── Mask cible ───────────────────────────────────────────────────
    mask_type: str = "mask_grid_major.png"

    # ── Architecture ─────────────────────────────────────────────────
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    # ── Résolution (Fixée à 1024) ────────────────────────────────────
    img_height: int = 1024
    img_width: int = 1024

    # ── Entraînement ─────────────────────────────────────────────────
    batch_size: int = 4
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    num_workers: int = 0
    pin_memory: bool = True

    # ── Loss & Scheduler ─────────────────────────────────────────────
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # ── Split ────────────────────────────────────────────────────────
    # On garde tes sources réelles (031, 032 pour train, 033 pour val)
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources: list = field(default_factory=lambda: ["ECG_033"])

    # ── Device ───────────────────────────────────────────────────────
    device: str = ""

    # ── Divers ───────────────────────────────────────────────────────
    seed: int = 42
    save_every_n_epochs: int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        # Auto-configurer les chemins
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.mask_dir:
            self.mask_dir = os.path.join(self.project_root, "output_augmentation", "masks")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs")

        # Détection du hardware sans modification des paramètres
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU détecté : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - Attention : les performances seront très limitées à 1024x1024.")

# Initialisation
cfg = TrainConfig()

In [ ]:
"""
Dataset PyTorch pour la segmentation de grille ECG.

Charge les paires (image augmentée P2, mask grille P2) et les prépare
pour l'entraînement d'un U-Net.
"""

import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import albumentations as A


class ECGGridDataset(Dataset):
    """
    Dataset pour la segmentation de grille sur ECG augmentés.

    Chaque sample = (image_augmentée, mask_grille) avec les deux
    dans l'espace P2 (déformé), alignés pixel-à-pixel.

    Args:
        image_dir: dossier avec les images augmentées (.webp)
        mask_dir: dossier parent des masks (contient un sous-dossier par image)
        mask_type: nom du fichier mask à utiliser comme cible
        source_prefixes: liste de préfixes pour filtrer (ex: ["ECG_031"])
                         si None, prend tout
        img_height, img_width: résolution de sortie
        augment: si True, applique des augmentations d'entraînement
    """

    def __init__(
        self,
        image_dir: str,
        mask_dir: str,
        mask_type: str = "mask_grid_major.png",
        source_prefixes: list = None,
        img_height: int = 1024,
        img_width: int = 1024,
        augment: bool = False,
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.mask_type = mask_type
        self.img_height = img_height
        self.img_width = img_width

        # Construire la liste des paires image/mask
        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue

            # Filtrer par source si demandé
            if source_prefixes:
                if not any(fname.startswith(p) for p in source_prefixes):
                    continue

            stem = fname.replace(".webp", "")
            mask_path = os.path.join(mask_dir, stem, mask_type)
            if os.path.exists(mask_path):
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "mask_path": mask_path,
                    "stem": stem,
                })

        # Augmentations d'entraînement
        # NOTE: ce sont des augmentations SUPPLÉMENTAIRES à celles de P2.
        # On n'applique que des transformations légères qui ne changent pas
        # la géométrie (sinon le mask ne correspond plus).
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                # Couleur uniquement (ne touche pas la géométrie)
                A.ColorJitter(
                    brightness=0.15, contrast=0.15,
                    saturation=0.1, hue=0.05, p=0.5
                ),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees"
              f" (mask: {mask_type},"
              f" sources: {source_prefixes or 'toutes'})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Charger l'image en RGB
        img = Image.open(sample["image_path"]).convert("RGB")
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0



        # Charger le mask en grayscale
        mask = Image.open(sample["mask_path"]).convert("L")
        mask = mask.resize((self.img_width, self.img_height), Image.NEAREST)
        mask_np = np.array(mask, dtype=np.float32) / 255.0  # [H, W] ∈ [0, 1]

        # Augmentations (appliquées conjointement à l'image et au mask)
        if self.transform:
            transformed = self.transform(image=img_np, mask=mask_np)
            img_np = transformed["image"]
            mask_np = transformed["mask"]

        # Convertir en tenseurs PyTorch [C, H, W]
        
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float()  # [3, H, W]
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()    # [1, H, W]

        return img_tensor, mask_tensor


In [ ]:
"""
Script d'entraînement U-Net pour la détection de grille ECG.

Usage:
    python training/train.py                    # Auto-détecte CPU/GPU
    python training/train.py --device cpu       # Force CPU
    python training/train.py --device cuda      # Force GPU
    python training/train.py --epochs 5         # Override rapide
    python training/train.py --mask mask_grid_combined.png  # Autre mask

Tout est loggé dans training/runs/<timestamp>/
"""

import os
import sys
import time
import argparse
import datetime
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # backend non-interactif
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# Ajouter le projet au path
sys.path.insert(0, os.path.abspath(""))

#from training.config import TrainConfig
#from training.dataset import ECGGridDataset


# ═══════════════════════════════════════════════════════════════════════
# Loss Functions
# ═══════════════════════════════════════════════════════════════════════

class DiceLoss(nn.Module):
    """Dice Loss pour segmentation binaire. Gère le déséquilibre de classes."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred_sig = torch.sigmoid(pred)
        intersection = (pred_sig * target).sum(dim=(2, 3))
        union = pred_sig.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    """Combinaison BCE + Dice (souvent la meilleure pour la segmentation)."""
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight

    def forward(self, pred, target):
        return (self.bce_weight * self.bce(pred, target)
                + (1 - self.bce_weight) * self.dice(pred, target))


def get_loss(loss_type, bce_weight=0.5):
    """Factory pour la loss function."""
    if loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    elif loss_type == "dice":
        return DiceLoss()
    elif loss_type == "bce_dice":
        return BCEDiceLoss(bce_weight)
    else:
        raise ValueError(f"Loss inconnue: {loss_type}")


# ═══════════════════════════════════════════════════════════════════════
# Métriques
# ═══════════════════════════════════════════════════════════════════════

def compute_metrics(pred, target, threshold=0.5):
    """Calcule IoU et Dice sur un batch."""
    with torch.no_grad():
        pred_bin = (torch.sigmoid(pred) > threshold).float()

        intersection = (pred_bin * target).sum(dim=(2, 3))
        pred_sum = pred_bin.sum(dim=(2, 3))
        target_sum = target.sum(dim=(2, 3))

        # Dice
        dice = (2 * intersection + 1e-6) / (pred_sum + target_sum + 1e-6)

        # IoU (Jaccard)
        union = pred_sum + target_sum - intersection
        iou = (intersection + 1e-6) / (union + 1e-6)

        # Pixel accuracy (sur les pixels de grille seulement)
        # = recall / sensitivity
        recall = (intersection + 1e-6) / (target_sum + 1e-6)

        # Precision
        precision = (intersection + 1e-6) / (pred_sum + 1e-6)

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


# ═══════════════════════════════════════════════════════════════════════
# Visualisation
# ═══════════════════════════════════════════════════════════════════════

def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    """
    Sauvegarde une grille de visualisation :
    [image | mask vrai | mask prédit | overlay]
    """
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mask_true = masks_true[i, 0].cpu().numpy()
        mask_pred = (torch.sigmoid(masks_pred[i, 0]).cpu().numpy() > 0.5).astype(float)

        # Image originale
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Image augmentée", fontsize=9)
        axes[i, 0].axis("off")

        # Mask vrai (ground truth)
        axes[i, 1].imshow(mask_true, cmap="gray", vmin=0, vmax=1)
        axes[i, 1].set_title("Mask vrai (GT)", fontsize=9)
        axes[i, 1].axis("off")

        # Mask prédit
        axes[i, 2].imshow(mask_pred, cmap="gray", vmin=0, vmax=1)
        axes[i, 2].set_title("Mask prédit", fontsize=9)
        axes[i, 2].axis("off")

        # Overlay : rouge = faux positif, vert = vrai positif, bleu = faux négatif
        overlay = img.copy()
        tp = (mask_pred > 0.5) & (mask_true > 0.5)
        fp = (mask_pred > 0.5) & (mask_true < 0.5)
        fn = (mask_pred < 0.5) & (mask_true > 0.5)
        overlay[tp] = [0, 1, 0]   # vert = bien détecté
        overlay[fp] = [1, 0, 0]   # rouge = faux positif
        overlay[fn] = [0, 0, 1]   # bleu = manqué
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9)
        axes[i, 3].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════
# Entraînement
# ═══════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Entraîne un epoch, retourne la loss moyenne et les métriques."""
    model.train()
    total_loss = 0
    total_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    n_batches = 0

    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)

        # Forward
        pred = model(images)
        loss = criterion(pred, masks)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Métriques
        metrics = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in total_metrics:
            total_metrics[k] += metrics[k]
        n_batches += 1

        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{metrics['dice']:.3f}")

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in total_metrics.items()}
    return avg_loss, avg_metrics


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Valide le modèle, retourne la loss moyenne et les métriques."""
    model.eval()
    total_loss = 0
    total_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    n_batches = 0

    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        pred = model(images)
        loss = criterion(pred, masks)

        metrics = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in total_metrics:
            total_metrics[k] += metrics[k]
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in total_metrics.items()}
    return avg_loss, avg_metrics


def train(cfg: TrainConfig):
    """Boucle d'entraînement principale."""

    # ── Setup ────────────────────────────────────────────────────────
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    # Sauvegarder la config
    config_dict = {k: str(v) if not isinstance(v, (int, float, bool, list, type(None)))
                   else v for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(config_dict, f, indent=2)

    print(f"\n{'='*60}")
    print(f"  Entrainement U-Net -- Detection de grille ECG")
    print(f"{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Mask cible  : {cfg.mask_type}")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}")
    print(f"{'='*60}\n")

    # Seed
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    device = torch.device(cfg.device)

    # ── Datasets ─────────────────────────────────────────────────────
    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  Masks:  {cfg.mask_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_dataset = ECGGridDataset(
        image_dir=cfg.image_dir,
        mask_dir=cfg.mask_dir,
        mask_type=cfg.mask_type,
        source_prefixes=cfg.train_sources,
        img_height=cfg.img_height,
        img_width=cfg.img_width,
        augment=True,
    )

    print(f"  Val (sources: {cfg.val_sources}):")
    val_dataset = ECGGridDataset(
        image_dir=cfg.image_dir,
        mask_dir=cfg.mask_dir,
        mask_type=cfg.mask_type,
        source_prefixes=cfg.val_sources,
        img_height=cfg.img_height,
        img_width=cfg.img_width,
        augment=False,
    )

    if len(train_dataset) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee !")
        print("   Vérifie que les images et masks existent dans les dossiers ci-dessus.")
        print("   Exécute d'abord les pipelines P1 et P2 pour générer les données.")
        return

    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )

    # ── Modèle ───────────────────────────────────────────────────────
    print("\n[*] Construction du modele...")
    import segmentation_models_pytorch as smp

    model = smp.Unet(
        encoder_name=cfg.encoder_name,
        encoder_weights=cfg.encoder_weights,
        in_channels=cfg.in_channels,
        classes=cfg.num_classes,
        activation=None,  # on applique sigmoid dans la loss
    )
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parametres: {total_params:,} (trainable: {trainable:,})")

    # ── Loss, Optimizer, Scheduler ───────────────────────────────────
    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience,
        factor=cfg.scheduler_factor,
    )

    # ── Historique ───────────────────────────────────────────────────
    history = {
        "train_loss": [], "val_loss": [],
        "train_dice": [], "val_dice": [],
        "train_iou": [], "val_iou": [],
        "lr": [],
    }
    best_val_dice = 0
    epochs_no_improve = 0

    # ── Boucle d'entraînement ────────────────────────────────────────
    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(1, cfg.num_epochs + 1):
        t_epoch = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        # Train
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validation
        val_loss, val_metrics = validate(model, val_loader, criterion, device)

        # Scheduler
        scheduler.step(val_loss)

        # Historique
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_dice"].append(train_metrics["dice"])
        history["val_dice"].append(val_metrics["dice"])
        history["train_iou"].append(train_metrics["iou"])
        history["val_iou"].append(val_metrics["iou"])
        history["lr"].append(current_lr)

        # Affichage
        elapsed = time.time() - t_epoch
        print(
            f"Epoch {epoch:3d}/{cfg.num_epochs} | "
            f"Train Loss: {train_loss:.4f}  Dice: {train_metrics['dice']:.3f} | "
            f"Val Loss: {val_loss:.4f}  Dice: {val_metrics['dice']:.3f}  "
            f"IoU: {val_metrics['iou']:.3f} | "
            f"LR: {current_lr:.1e} | {elapsed:.1f}s"
        )

        # Sauvegarder le meilleur modèle
        if val_metrics["dice"] > best_val_dice:
            best_val_dice = val_metrics["dice"]
            epochs_no_improve = 0
            best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": best_val_dice,
                "config": config_dict,
            }, best_path)
            print(f"  [BEST] Nouveau meilleur modele sauvegarde (Dice: {best_val_dice:.4f})")
        else:
            epochs_no_improve += 1

        # Checkpoint périodique
        if epoch % cfg.save_every_n_epochs == 0:
            ckpt_path = os.path.join(
                run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"
            )
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_metrics["dice"],
            }, ckpt_path)

        # Visualisation périodique
        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                # Prendre le premier batch de validation
                sample_images, sample_masks = next(iter(val_loader))
                sample_pred = model(sample_images.to(device)).cpu()
                vis_path = os.path.join(
                    run_dir, "visualizations", f"epoch_{epoch:03d}.png"
                )
                save_prediction_grid(sample_images, sample_masks, sample_pred, vis_path)

        # Early stopping
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration")
            break

    # ── Fin ───────────────────────────────────────────────────────────
    total_time = time.time() - t_start
    print(f"\n{'='*60}")
    print(f"  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}")
    print(f"{'='*60}")

    # Sauvegarder l'historique
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Courbes d'entraînement
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))

    return run_dir


def _plot_training_curves(history, save_path):
    """Trace les courbes de loss et Dice."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    ax1.plot(epochs, history["train_loss"], "b-", label="Train")
    ax1.plot(epochs, history["val_loss"], "r-", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss (BCE + Dice)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Dice
    ax2.plot(epochs, history["train_dice"], "b-", label="Train Dice")
    ax2.plot(epochs, history["val_dice"], "r-", label="Val Dice")
    ax2.plot(epochs, history["train_iou"], "b--", alpha=0.5, label="Train IoU")
    ax2.plot(epochs, history["val_iou"], "r--", alpha=0.5, label="Val IoU")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.set_title("Dice & IoU")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [PLOT] Courbes sauvegardees: {save_path}")


# ═══════════════════════════════════════════════════════════════════════
# CLI
# ═══════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description="Entraîner un U-Net pour la détection de grille ECG"
    )
    parser.add_argument("--device", type=str, default="",
                        help="Device: 'cpu' ou 'cuda' (auto si vide)")
    parser.add_argument("--epochs", type=int, default=None,
                        help="Nombre d'epochs (override)")
    parser.add_argument("--batch-size", type=int, default=None,
                        help="Taille du batch (override)")
    parser.add_argument("--lr", type=float, default=None,
                        help="Learning rate (override)")
    parser.add_argument("--resolution", type=int, default=None,
                        help="Résolution carrée (ex: 512)")
    parser.add_argument("--mask", type=str, default=None,
                        help="Type de mask (ex: mask_grid_major.png)")
    parser.add_argument("--encoder", type=str, default=None,
                        help="Encoder (ex: resnet34, resnet50)")
    args, unknown = parser.parse_known_args()

    # Construire la config avec les overrides
    overrides = {}
    if args.device:
        overrides["device"] = args.device
    if args.mask:
        overrides["mask_type"] = args.mask

    cfg = TrainConfig(**overrides)

    # Overrides post-init (après l'ajustement CPU/GPU automatique)
    if args.epochs is not None:
        cfg.num_epochs = args.epochs
    if args.batch_size is not None:
        cfg.batch_size = args.batch_size
    if args.lr is not None:
        cfg.learning_rate = args.lr
    if args.resolution is not None:
        cfg.img_height = args.resolution
        cfg.img_width = args.resolution
    if args.encoder is not None:
        cfg.encoder_name = args.encoder

    train(cfg)


if __name__ == "__main__":
    main()


In [ ]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp
from IPython.display import display

# Trouver automatiquement le dernier run
runs_dir = cfg.output_dir  # /content/drive/MyDrive/data/training/runs

run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"❌ Aucun run trouvé dans {runs_dir}")

run_dir = run_dirs[-1]  # tri alphabétique = tri chronologique
print(f"✅ Run sélectionné : {run_dir}")

# Priorité : best_model > dernier checkpoint périodique
best_model_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
all_checkpoints = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))

if os.path.exists(best_model_path):
    checkpoint_path = best_model_path
    print(f"🏆 Meilleur modèle : {checkpoint_path}")
elif all_checkpoints:
    checkpoint_path = all_checkpoints[-1]
    print(f"📦 Dernier checkpoint : {checkpoint_path}")
else:
    raise FileNotFoundError(f"❌ Aucun checkpoint trouvé dans {run_dir}/checkpoints/")

# Charger le modèle
DEVICE = cfg.device

model = smp.Unet(
    encoder_name=cfg.encoder_name,
    encoder_weights=None,
    in_channels=cfg.in_channels,
    classes=cfg.num_classes,
)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()
print(f"✅ Modèle chargé — epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")

In [ ]:
image_dir = cfg.image_dir

val_images   = sorted([f for f in os.listdir(image_dir) if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir) if (f.startswith("ECG_031") or f.startswith("ECG_032")) and f.endswith(".webp")])



In [ ]:
# ← MODIFIER ICI
%matplotlib inline
SET   = "val"   # "val" ou "train"
INDEX = 567
    # index dans la liste ci-dessus

# Sélection
image_list = val_images if SET == "val" else train_images

if INDEX >= len(image_list):
    raise IndexError(f"❌ INDEX={INDEX} hors limites — {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
mask_path   = os.path.join(cfg.mask_dir, sample_file.replace(".webp", ""), cfg.mask_type)

print(f"Image  : {sample_file}")
print(f"Mask   : {mask_path}")
print(f"Mask existe : {os.path.exists(mask_path)}")

if not os.path.exists(mask_path):
    raise FileNotFoundError(f"❌ Mask introuvable : {mask_path}")

# Prétraitement — image (identique au training : /255 SEULEMENT, pas d'ImageNet norm)
img_pil = Image.open(sample_path).convert("RGB")
img_pil = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0

# Prétraitement — mask
mask_pil = Image.open(mask_path).convert("L")
mask_pil = mask_pil.resize((cfg.img_width, cfg.img_height), Image.NEAREST)
mask_np  = np.array(mask_pil, dtype=np.float32) / 255.0

# Tenseur
img_tensor = (
    torch.from_numpy(img_np)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .to(DEVICE)
)

# Inférence
with torch.no_grad():
    output     = model(img_tensor)
    prediction = torch.sigmoid(output)

pred_np     = prediction.squeeze().cpu().numpy()
pred_binary = (pred_np > 0.5).astype(np.float32)

# Dice
intersection = (pred_binary * mask_np).sum()
dice = (2 * intersection) / (pred_binary.sum() + mask_np.sum() + 1e-8)

print(f"\nMax: {pred_np.max():.4f} | Min: {pred_np.min():.4f} | Moyenne: {pred_np.mean():.4f}")
print(f"Dice sur cette image : {dice:.4f}")

# Overlay TP/FP/FN (sur l'image brute non normalisée)
overlay = img_np.copy()
tp = (pred_binary > 0.5) & (mask_np > 0.5)
fp = (pred_binary > 0.5) & (mask_np < 0.5)
fn = (pred_binary < 0.5) & (mask_np > 0.5)
overlay[tp] = [0.0, 1.0, 0.0]   # vert  = vrai positif
overlay[fp] = [1.0, 0.0, 0.0]   # rouge = faux positif
overlay[fn] = [0.0, 0.0, 1.0]   # bleu  = faux négatif

# Visualisation
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f"[{SET.upper()}]  {sample_file}  —  Dice: {dice:.4f}", fontsize=12)

axes[0].imshow(img_np)
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].imshow(mask_np, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Mask cible (vérité terrain)")
axes[1].axis("off")

axes[2].imshow(pred_np, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Prédiction brute (sigmoid)")
axes[2].axis("off")

axes[3].imshow(overlay)
axes[3].set_title("Overlay (Vert=TP, Rouge=FP, Bleu=FN)")
axes[3].axis("off")

plt.tight_layout()
plt.show()
plt.close()
